# U.S. Aviation COVID-19 Impact Analysis

## SQL Exploratory Data Analysis

**Objective:** Investigate how U.S. aviation activity changed between 2019 (pre-pandemic) and 2020 (pandemic period), focusing on flight volumes, airline operations, cancellations, and delays.

**Database:** PostgreSQL

**Dataset:** U.S. Flight Records (2019–2020)

**Tables:**
- `flights_2019_new`
- `flights_2020_new`

**Research Questions:**
1. How did monthly flight volumes change between 2019 and 2020?
2. Which airlines operated the most flights?
3. How did cancellation rates change?
4. Which airlines experienced the highest cancellation rates?
5. How did departure and arrival delays change?
6. Which airports experienced the greatest changes in flight activity?
7. What patterns emerged during the COVID-19 disruption?

**Data limitation:** The datasets are samples of flight records. Results describe the available samples and should not automatically be interpreted as complete nationwide flight totals.

In [ ]:
%load_ext sql
%sql engine

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

password = getpass("PostgreSQL password: ")

db_url = URL.create(
    "postgresql+psycopg",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="aviation_covid_analysis"
)

engine = create_engine(db_url)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database()"))
    print("Connected to:", result.scalar())

In [ ]:
%load_ext sql
%sql engine

In [ ]:
%%sql

SELECT * 
FROM flights_2019;

SELECT * 
FROM flights_2020;

In [ ]:
%%sql

SELECT 
    2019 AS year, 
    COUNT(*) AS total_flights
FROM flights_2019

UNION ALL

SELECT
    2020 AS year,
    COUNT(*) AS total_flights
FROM flights_2020;

In [ ]:
%%sql
# date range in each dataset
SELECT 
    MIN(fl_date) AS min_date_2019,
    MAX(fl_date) AS max_date_2019
FROM flights_2019_new;

SELECT
    MIN(fl_date) AS min_date_2020,
    MAX(fl_date) AS max_date_2020
FROM flights_2020_new;

In [ ]:
%%sql 

SELECT 
    COUNT(DISTINCT EXTRACT(MONTH FROM fl_date)) AS distinct_month
FROM 
    flights_2020_new;
    
# oraclesql or mysql
# SELECT 
#     MONTH(fl_date) AS month
# FROM 
#     flights_2020_new;

In [ ]:
%%sql

SELECT
    EXTRACT(YEAR FROM fl_date) AS year,
    COUNT(DISTINCT EXTRACT(MONTH FROM fl_date)) AS distinct_months
FROM flights_2019_new
GROUP BY EXTRACT(YEAR FROM fl_date)

UNION ALL

SELECT
    EXTRACT(YEAR FROM fl_date) AS year,
    COUNT(DISTINCT EXTRACT(MONTH FROM fl_date)) AS distinct_months
FROM flights_2020_new
GROUP BY EXTRACT(YEAR FROM fl_date)

ORDER BY year;

In [ ]:
%%sql

SELECT 
    2019 AS year,
    COUNT(*) AS total_flights
FROM flights_2019_new
WHERE dep_delay IS NULL
GROUP BY year

UNION ALL

SELECT 
    2020 AS year,
    COUNT(*) AS total_flights
FROM flights_2020_new
WHERE dep_delay IS NULL
GROUP BY year;

In [ ]:
%%sql

SELECT
    2019 AS year,
    cancelled,
    COUNT(*) AS total_flights
FROM flights_2019_new
GROUP BY cancelled

UNION ALL

SELECT
    2020 AS year,
    cancelled,
    COUNT(*) AS total_flights
FROM flights_2020_new
GROUP BY cancelled

ORDER BY year, cancelled;

In [ ]:
%%sql

SELECT
    2019 AS year,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled_flights,
    ROUND(
    100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)
    / COUNT(*),
    2
) AS cancellation_rate

FROM flights_2019_new;

In [ ]:
%%sql

SELECT 
    2020 AS year,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled_flights,
    ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/ COUNT(*), 2) AS cancellation_rate
FROM flights_2020_new;

In [ ]:
%%sql

--montly cancellation analysis

SELECT
    EXTRACT(MONTH FROM fl_date) AS month,
    2020 AS year,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled_flight,
    ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancellation_rate
FROM flights_2020_new
GROUP BY month
ORDER BY month;

In [ ]:
%%sql

--highest cancellation rate

SELECT 
    EXTRACT(MONTH FROM fl_date) AS month,
    2020 AS year,
    COUNT(*) AS total_flight,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled_flights,
    ROUND(100*0 SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancellation_rate
FROM flights_2020_new
GROUP BY month
ORDER BY cancellation_rate DESC
LIMIT 1;

    

In [ ]:
%%sql

-- monthly cancellation rate of all 2 years

SELECT
    EXTRACT(MONTH FROM fl_date) AS month,
    2019 AS year,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS total_cancelled_flights,
    ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*), 2 ) AS cancelled_percent
FROM flights_2019_new
GROUP BY month

UNION ALL

SELECT  
    EXTRACT(MONTH FROM fl_date) AS month,
    2020 AS year,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END ) AS total_cancelled_flights,
    ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END )/COUNT(*),2 ) AS cancelled_percent
FROM flights_2020_new
GROUP BY month
ORDER BY year,month;

    

In [ ]:
%%sql
WITH firstyear AS (
    SELECT 
        2019 AS year,
        EXTRACT(MONTH FROM fl_date) AS month,
        COUNT(*) AS total_flights,
        SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS total_cancelled,
        ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancelled_rate
    FROM flights_2019_new
    GROUP BY month
),
secondyear AS (
    SELECT 
        2020 AS year,
        EXTRACT(MONTH FROM fl_date) AS month,
        COUNT(*) AS total_flights,
        SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS total_cancelled,
        ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancelled_rate
    FROM flights_2020_new
    GROUP BY month
)

SELECT 
    y1.month,
    y1.cancelled_rate,
    y2.cancelled_rate,
    (y2.cancelled_rate - y1.cancelled_rate) AS percent_point_difference
FROM firstyear y1
JOIN secondyear y2
    ON y1.month = y2.month


In [ ]:
%%sql
WITH firstyear AS (
    SELECT 
        2019 AS year,
        EXTRACT(MONTH FROM fl_date) AS month,
        COUNT(*) AS total_flights,
        SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS total_cancelled,
        ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancelled_rate
    FROM flights_2019_new
    GROUP BY month
),
secondyear AS (
    SELECT 
        2020 AS year,
        EXTRACT(MONTH FROM fl_date) AS month,
        COUNT(*) AS total_flights,
        SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS total_cancelled,
        ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancelled_rate
    FROM flights_2020_new
    GROUP BY month
)

SELECT 
    y1.month,
    y1.cancelled_rate,
    y2.cancelled_rate,
    (y2.cancelled_rate - y1.cancelled_rate) AS percent_point_difference
FROM firstyear y1
JOIN secondyear y2
    ON y1.month = y2.month
ORDER BY percent_point_difference DESC
LIMIT 1;

In [ ]:
%%sql
--10 airlines with the highest cancellation rates in 2020

SELECT 
    2020 AS year,
    airline,
    COUNT(*) AS total_flight,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS total_cancelled,
    ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*),2) AS cancelled_rate
FROM flights_2020_new
GROUP BY airline
ORDER BY cancelled_rate DESC
LIMIT 10;
    


In [ ]:
%%sql

-- airline difference

WITH secondyear AS (
    SELECT 
        2020 AS year,
        airline,
        COUNT(*) AS total_flights,
        SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled,
        ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancelled_rate
    FROM flights_2020_new
    GROUP BY airline
    
),
firstyear AS(
    SELECT 
        2019 AS year,
        airline,
        COUNT(*) AS total_flights,
        SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled,
        ROUND(100.0 * SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancelled_rate
    FROM flights_2019_new
    GROUP BY airline
)
SELECT 
    y1.airline,
    y1.cancelled_rate AS cancelled_rate_2019,
    y2.cancelled_rate AS cancelled_rate_2020,
    (y2.cancelled_rate - y1.cancelled_rate) AS cancelled_rate
FROM firstyear y1
JOIN secondyear y2
    ON y1.airline = y2.airline
ORDER BY cancelled_rate DESC;


In [ ]:
%%sql

SELECT
    2019 AS year,
    origin,
    COUNT(*) AS total_flights,
    RANK() OVER (
        ORDER BY COUNT(*) DESC
    ) AS airport_rank
FROM flights_2019_new
GROUP BY origin
ORDER BY airport_rank
LIMIT 10;

In [ ]:
%%sql

SELECT 
    2020 AS year,
    origin,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END ) AS cancelled_flights,
    ROUND(SUM(CASE WHEN cancelled= 1 THEN 1 ELSE 0 END)*100.0 / COUNT(*), 2) AS cancellation_rate
FROM flights_2020_new
GROUP BY origin
ORDER BY cancellation_rate DESC

In [ ]:
%%sql

SELECT 
    2019 AS year, 
    origin,
    COUNT(*) AS total_flights, 
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS flight_cancelled,
    ROUND(SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)* 100.0/ COUNT(*), 2) AS cancellation_rate,
    RANK() OVER(ORDER BY SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)*100.0 / COUNT(*) DESC) AS airport_rank
FROM flights_2019_new
GROUP BY origin 
HAVING COUNT(*) >=100
ORDER BY airport_rank ASC
LIMIT 10;

 

In [ ]:
%%sql

SELECT 
    2020 AS year, 
    origin,
    COUNT(*) AS total_flights, 
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS flight_cancelled,
    ROUND(SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)* 100.0/ COUNT(*), 2) AS cancellation_rate,
    RANK() OVER(ORDER BY SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)*100.0 / COUNT(*) DESC) AS airport_rank
FROM flights_2020_new
GROUP BY origin 
HAVING COUNT(*) >=100
ORDER BY airport_rank ASC
LIMIT 10;

In [ ]:
%%sql

-- compare cancellation rate

WITH first_year AS(
    SELECT
        origin,
        2019 AS year,
        COUNT(*) AS total_flight,
        SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled_flights,
        ROUND(SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)*100.0 / COUNT(*),2) cancelled_flights_rate
    FROM flights_2019_new
    GROUP BY origin
    HAVING COUNT(*) >= 100
    ORDER BY cancelled_flights_rate DESC
),
second_year AS(
    SELECT
        origin,
        2020 AS year,
        COUNT(*) AS total_flight,
        SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled_flights,
        ROUND(SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END)*100.0 / COUNT(*),2) cancelled_flights_rate
    FROM flights_2020_new
    GROUP BY origin
    HAVING COUNT(*) >=100
    ORDER BY cancelled_flights_rate DESC
)

SELECT 
    y1.origin,
    y1.cancelled_flights_rate AS year_2019,
    y2.cancelled_flights_rate AS year_2020,
    (y2.cancelled_flights_rate - y1.cancelled_flights_rate) AS difference_in_rate
FROM first_year y1
JOIN second_year y2
    ON y1.origin = y2.origin
ORDER BY difference_in_rate DESC
LIMIT 10
;
        

In [ ]:
%%sql

SELECT 
    2019 AS year,
    COUNT(*) AS total_flight,
    ROUND(AVG(dep_delay)::NUMERIC,2) AS average_departure_delay
FROM flights_2019_new
WHERE cancelled = 0

UNION ALL

SELECT 
    2020 AS year, 
    COUNT(*) AS total_flight,
    ROUND(AVG(dep_delay)::NUMERIC, 2) AS average_departure_delay
FROM flights_2020_new
WHERE cancelled = 0;

In [ ]:
%%sql

--monthly departure delay analysis

SELECT 
    2020 AS year,
    EXTRACT(MONTH FROM fl_date) AS month,
    COUNT(*) AS total_flights,
    ROUND(AVG(dep_delay)::NUMERIC,2) AS average_departure_delay
FROM flights_2020_new
WHERE cancelled = 0
GROUP BY month
ORDER BY average_departure_delay DESC;


In [ ]:
%%sql

--comparing monthly departure delays between 2019 and 2020
WITH firstyear AS(
SELECT 
    EXTRACT(MONTH FROM fl_date) AS month,
    COUNT(*) AS total_flight,
    ROUND(AVG(dep_delay)::NUMERIC,2) AS departure_delay
FROM flights_2019_new
WHERE cancelled = 0 
GROUP BY month
),
secondyear AS(
SELECT 
    EXTRACT(MONTH FROM fl_date) AS month,
    COUNT(*) AS total_flight,
    ROUND(AVG(dep_delay)::NUMERIC,2) AS departure_delay
FROM flights_2020_new
WHERE cancelled = 0
GROUP BY month
)

SELECT 
    y1.month,
    y1.departure_delay AS average_delay_2019,
    y2.departure_delay AS average_delay_2020,
    (y2.departure_delay - y1.departure_delay) AS total_departure_delay
FROM firstyear y1
JOIN secondyear y2 
    ON y1.month = y2.month
ORDER BY total_departure_delay DESC
;

In [ ]:
%%sql

--Airline departure delay performance
WITH firstyear AS(
SELECT
    airline,
    COUNT(*) AS total_flight,
    ROUND(AVG(dep_delay)::NUMERIC, 2) AS average_delay
FROM flights_2019_new
WHERE cancelled = 0
GROUP BY airline
HAVING COUNT(*) >=100
),
secondyear AS
(
SELECT
    airline,
    COUNT(*) AS total_flight,
    ROUND(AVG(dep_delay)::NUMERIC, 2) AS average_delay
FROM flights_2020_new
WHERE cancelled = 0
GROUP BY airline
HAVING COUNT(*) >=100
)
SELECT 
    f1.airline,
    f1.average_delay,
    f2.average_delay,
    f2.average_delay - f1.average_delay AS total_average_delay
FROM firstyear f1
JOIN secondyear f2
    ON f1.airline = f2.airline
ORDER BY total_average_delay DESC
LIMIT 10;